<a href="https://colab.research.google.com/github/Edu0802/pyspark-bigdata-analysis/blob/main/Atividade_2_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Questão 1 (código)

In [2]:
# Bloco de Preparação do Ambiente
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/nyc_tripdata_2024_sample_4M.csv

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("ExerciciosPySpark")
    .master("local[*]")
    .getOrCreate()
)

# Carregamento do DataFrame principal
df = spark.read.csv("nyc_tripdata_2024_sample_4M.csv", header=True, inferSchema=True)

# Questão 2 (código)
Selecione apenas as colunas VendorID, tpep_pickup_datetime, trip_distance, fare_amount e payment_type, e exiba as 5 primeiras linhas do resultado.

In [3]:
# Seleção das colunas solicitadas
df_selecionado = df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "payment_type"
)

# Exibição das 5 primeiras linhas
df_selecionado.show(5)

+--------+--------------------+-------------+-----------+------------+
|VendorID|tpep_pickup_datetime|trip_distance|fare_amount|payment_type|
+--------+--------------------+-------------+-----------+------------+
|       1| 2024-10-01 00:59:55|          0.5|        5.1|           1|
|       1| 2024-10-01 00:08:59|         20.6|       76.5|           2|
|       2| 2024-10-01 00:18:38|         7.42|       33.1|           4|
|       2| 2024-10-01 00:20:06|        19.96|       70.0|           1|
|       1| 2024-10-01 00:09:02|          2.6|       15.6|           1|
+--------+--------------------+-------------+-----------+------------+
only showing top 5 rows


# Questão 3 (código)
Filtre as corridas que atendem simultaneamente às duas condições abaixo, e exiba quantas corridas restaram:
- trip_distance maior que 5 milhas;
- passenger_count maior ou igual a 3.

In [4]:
# Filtragem das corridas atendendo a ambas as condições simultaneamente
df_filtrado = df.filter(
    (df["trip_distance"] > 5) & (df["passenger_count"] >= 3)
)

# Contagem e exibição do número de corridas resultantes
total_corridas = df_filtrado.count()
print(f"Quantidade de corridas que atendem aos critérios: {total_corridas}")

Quantidade de corridas que atendem aos critérios: 50665


# Questão 4 (descritiva)

## Funcionamento da Inferência de Schema (`inferSchema=True`)
Quando definimos `inferSchema=True`, o Spark precisa realizar um **passo adicional de leitura prévia dos dados** para tentar adivinhar o tipo de dado de cada coluna (se é do tipo `Integer`, `Double`, `Timestamp`, `String`, etc.). Para fazer isso em um arquivo CSV, o Spark lê uma amostra significativa das linhas do arquivo, inspeciona o formato dos valores presentes em cada coluna e escolhe o tipo de dado compatível mais específico.

---

## Comparação de Abordagens: `inferSchema=True` vs. Schema Manual (`StructType`/`StructField`)

| Aspecto | Inferência Automática (`inferSchema=True`) | Schema Manual (`StructType` / `StructField`) |
| :--- | :--- | :--- |
| **Gargalo e Performance** | **Lento para grandes arquivos.** Exige um *job* extra antes da leitura para analisar o conteúdo do arquivo (duas passagens pelo disco). | **Muito Rápido.** Nenhuma passagem prévia pelos dados é necessária. O Spark inicia a leitura/processamento imediatamente. |
| **Confiabilidade de Tipos** | **Sujeito a erros.** Se as linhas iniciais não representarem todo o arquivo, o Spark pode inferir tipos incorretos (ex: tratar uma coluna numérica como string ou vice-versa). | **Determinístico e Garantido.** O desenvolvedor define com precisão os tipos de dados e a aceitação de valores nulos (`nullable`). |
| **Conveniência de Uso** | **Alta.** Ideal para exploração rápida e prototipação de dados em ambiente de desenvolvimento/análise ad-hoc. | **Baixa em protótipos.** Requer escrever mais código inicial para definir a estrutura completa de todas as colunas. |
| **Ambiente Recomendado** | Análise exploratória e *notebooks* de testes com pequenos volumes de dados. | Ambientes de **Produção (Pipelines de ETL/ELT)** e processamento de Big Data em escala (milhões/bilhões de linhas). |

---

## Vantagens e Riscos no Contexto do Exercício (Base de 4 Milhões de Linhas)

* **Inferência (`inferSchema=True`):**
  * **Vantagem:** Facilidade e agilidade inicial, pois evita o trabalho manual de mapear as dezenas de colunas do dataset de táxis de NYC.
  * **Risco:** Perda expressiva de desempenho devido à leitura prévia necessária para determinar o *schema*. Além disso, dados inconsistentes ou pontualmente mal formatados em um arquivo com 4M de linhas podem fazer o Spark inferir uma coluna inteira como `String`, quebrando transformações posteriores de data ou cálculo numérico.

* **Schema Manual (`StructType`):**
  * **Vantagem:** Alta performance, previsibilidade total e resiliência. O Spark ignora a fase de varredura prévia do CSV e carrega os dados com segurança para o *pipeline* de produção.
  * **Risco:** Exige esforço de desenvolvimento prévio para codificar o *schema* e pode falhar ou gerar valores nulos caso o formato da fonte de dados mude sem aviso prévio (*schema drift*).



# Questão 5 (código)
Agrupe as corridas por `payment_type` e calcule, para cada grupo:
- A quantidade de corridas;
- A soma total de `total_amount` (receita total).

Exiba o resultado ordenado pela receita total, da maior para a menor.

In [5]:
# Importação das funções do Spark
from pyspark.sql import functions as F

# Agrupamento, agregações e ordenação
df_pagamentos = (
    df.groupBy("payment_type")
    .agg(
        F.count("*").alias("qtd_corridas"),
        F.sum("total_amount").alias("receita_total")
    )
    .orderBy(F.col("receita_total").desc())
)

# Exibição do resultado
df_pagamentos.show()

+------------+------------+--------------------+
|payment_type|qtd_corridas|       receita_total|
+------------+------------+--------------------+
|           1|     3045849| 9.116799616010016E7|
|           2|      553536|1.2987084559999354E7|
|           0|      410746|1.0123049400000528E7|
|           3|       29100|  220775.24999999974|
|           4|       79511|  133192.01999999984|
|           5|           1|                62.0|
+------------+------------+--------------------+



# Questão 6 (código)
Crie uma nova coluna chamada `hora_embarque`, extraindo apenas a hora (0 a 23) da coluna `tpep_pickup_datetime`. Em seguida, agrupe por `hora_embarque` e calcule a tarifa média (`fare_amount`) e a distância média (`trip_distance`) para cada hora do dia. Exiba o resultado ordenado pela hora, de 0 a 23.

In [6]:
from pyspark.sql import functions as F

# 1. Criação da coluna 'hora_embarque' extraindo a hora do timestamp de pickup
df_com_hora = df.withColumn("hora_embarque", F.hour("tpep_pickup_datetime"))

# 2. Agrupamento por hora, cálculo das médias e ordenação de 0 a 23
df_medias_por_hora = (
    df_com_hora.groupBy("hora_embarque")
    .agg(
        F.round(F.avg("fare_amount"), 2).alias("fare_amount_medio"),
        F.round(F.avg("trip_distance"), 2).alias("trip_distance_media")
    )
    .orderBy("hora_embarque")
)

# 3. Exibição das 24 horas do dia
df_medias_por_hora.show(24)

+-------------+-----------------+-------------------+
|hora_embarque|fare_amount_medio|trip_distance_media|
+-------------+-----------------+-------------------+
|            0|            19.73|               5.13|
|            1|            17.55|               3.74|
|            2|            16.43|               4.54|
|            3|            17.24|                3.4|
|            4|            22.34|              11.41|
|            5|            26.23|              23.34|
|            6|            21.93|              14.54|
|            7|            19.33|              11.09|
|            8|            18.51|               8.53|
|            9|             18.4|                5.6|
|           10|            18.56|               4.51|
|           11|            18.85|               4.08|
|           12|            19.21|               4.47|
|           13|            19.96|               5.26|
|           14|             20.6|               4.62|
|           15|            2

# Questão 7 (descritiva)

## Diferença entre Transformações e Ações
No Apache Spark, os comandos executados em um DataFrame pertencem a duas categorias fundamentais:

1. **Transformações (*Transformations*):**
   * **Conceito:** São operações que criam um novo DataFrame a partir de um existente, sem modificar os dados originais.
   * **Exemplos das questões anteriores:**
     * `select()` (Questão 2)
     * `filter()` (Questão 3)
     * `groupBy()`, `agg()` e `orderBy()` (Questão 5)
   * **Comportamento:** Não disparam o processamento imediato dos dados na CPU/Memória. Elas apenas registram o plano de execução no **DAG (Grafo Acíclico Dirigido)**.

2. **Ações (*Actions*):**
   * **Conceito:** São instruções que exigem que o Spark retorne um resultado concreto para o programa principal (Driver) ou grave dados em um armazenamento externo.
   * **Exemplos das questões anteriores:**
     * `.show()` (Questões 2 e 5)
     * `.count()` (Questão 3)
   * **Comportamento:** Ocorrem de forma "ansiosa" (*eager*). Quando uma ação é chamada, o Spark compila e executa fisicamente todo o fluxo de transformações acumulado no DAG.

---

## O Conceito de Avaliação Preguiçosa (*Lazy Evaluation*)
Diz-se que o Spark utiliza **Avaliação Preguiçosa (*Lazy Evaluation*)** porque ele adia ao máximo a execução real do processamento. O Spark não processa linha por linha à medida que os comandos de transformação são escritos; ele apenas constrói o plano lógico de etapas e só realiza o cálculo computacional no exato momento em que uma **Ação** (como `.show()`, `.count()` ou `.write`) é invocada.

---

## Vantagens Práticas do *Lazy Evaluation*

* **Otimização do Plano de Execução (*Catalyst Optimizer*):** Ao conhecer todo o fluxo de transformações antes de rodar, o Spark consegue otimizar as etapas. Por exemplo, se você aplicar um `filter()` no final do script, o Spark pode mover esse filtro para o início do processamento (*predicate pushdown*), evitando ler do disco milhões de linhas desnecessárias.
* **Redução de E/S (*I/O*) e Uso de Memória:** O Spark evita a criação de tabelas e resultados intermediários temporários no disco ou na RAM a cada linha de código.
* **Tolerância a Falhas Eficiente:** Como o Spark guarda o histórico exato das transformações (o DAG), caso um nó do *cluster* falhe durante a execução de uma Ação, ele consegue reconstruir apenas a parte dos dados perdida reexecutando as transformações necessárias para aquele bloco específico.

# Questão 8 (código)
Considerando apenas as corridas em que `total_amount` seja maior que zero, crie uma nova coluna chamada `percentual_gorjeta`, calculada como `(tip_amount / total_amount) * 100`. Em seguida, exiba as 10 corridas com maior `percentual_gorjeta`, mostrando as colunas `VendorID`, `total_amount`, `tip_amount` e `percentual_gorjeta`.

In [7]:
from pyspark.sql import functions as F

# 1. Filtragem para considerar apenas total_amount > 0
df_gorjetas = df.filter(F.col("total_amount") > 0)

# 2. Criação da coluna percentual_gorjeta
df_gorjetas = df_gorjetas.withColumn(
    "percentual_gorjeta",
    F.round((F.col("tip_amount") / F.col("total_amount")) * 100, 2)
)

# 3. Seleção das colunas solicitadas e ordenação pelas 10 maiores gorjetas
df_top10_gorjetas = (
    df_gorjetas.select(
        "VendorID",
        "total_amount",
        "tip_amount",
        "percentual_gorjeta"
    )
    .orderBy(F.col("percentual_gorjeta").desc())
)

# Exibição dos 10 primeiros resultados
df_top10_gorjetas.show(10)

+--------+------------+----------+------------------+
|VendorID|total_amount|tip_amount|percentual_gorjeta|
+--------+------------+----------+------------------+
|       2|        1.63|      5.27|            323.31|
|       2|        2.07|      3.68|            177.78|
|       2|         1.6|      2.82|            176.25|
|       2|        2.33|      3.72|            159.66|
|       2|        2.54|      3.76|            148.03|
|       2|        3.76|      3.96|            105.32|
|       2|        39.7|      40.0|            100.76|
|       2|        0.08|      0.08|             100.0|
|       1|       197.0|     196.0|             99.49|
|       1|       150.0|     149.0|             99.33|
+--------+------------+----------+------------------+
only showing top 10 rows


# Questão 9 (código)
Baixe a tabela de referência de zonas de táxi e faça o join para identificar os bairros de origem (*Borough*) de cada corrida:

a) Faça o join entre `df` e `zonas`, relacionando `df.PULocationID` com `zonas.LocationID`;

b) Agrupe o resultado por `Borough` e conte quantas corridas tiveram origem em cada um;

c) Exiba o resultado ordenado do bairro com mais corridas para o com menos.

In [8]:
# 1. Download e ingestão do arquivo de lookup das zonas
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/taxi_zone_lookup.csv

zonas = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

# Visualização inicial da tabela de referência
zonas.show(5)

# Importação de funções e realização do Join (a)
from pyspark.sql import functions as F

df_com_bairro = df.join(
    zonas,
    df["PULocationID"] == zonas["LocationID"],
    how="inner"
)

# Agrupamento por Borough, contagem de corridas (b) e ordenação decrescente (c)
df_corridas_por_bairro = (
    df_com_bairro.groupBy("Borough")
    .agg(F.count("*").alias("total_corridas"))
    .orderBy(F.col("total_corridas").desc())
)

# Exibição dos resultados
df_corridas_por_bairro.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
+-------------+--------------+
|      Borough|total_corridas|
+-------------+--------------+
|    Manhattan|       3641752|
|       Queens|        388736|
|     Brooklyn|         60200|
|        Bronx|         12702|
|      Unknown|         12172|
|          N/A|          2421|
|          EWR|           565|
|Staten Island|           195|
+-------------+--------------+



# Questão 10 (descritiva)

## Por que Operações de Agrupamento (`groupBy`) são mais Custosas que Filtragens (`filter`) e Seleções (`select`)

A diferença de desempenho entre uma contagem simples/filtragem e uma operação de agrupamento (`groupBy`) reside no impacto na rede e no movimento de dados entre os nós do *cluster* (ou entre as threads de execução do Spark), conceito conhecido como **Shuffle**.

---

## 1. Operações do Tipo *Narrow Dependency* (Sem Shuffle)
Operações como `select()`, `filter()` ou um `count()` global simples possuem **dependências estreitas (*Narrow Dependencies*)**:
* **Processamento Local:** O Spark consegue processar cada partição do arquivo CSV de forma totalmente independente e isolada na memória local do trabalhador (*Executor*).
* **Sem Troca de Rede:** Para filtrar uma linha (`trip_distance > 5`) ou selecionar colunas, um nó não precisa saber o que está acontecendo nos outros nós. As partições não conversam entre si, eliminando o gargalo de rede/disco.

---

## 2. Operações do Tipo *Wide Dependency* (Com Shuffle)
Operações como `groupBy("payment_type")` possuem **dependências amplas (*Wide Dependencies*)**:
* **Redistribuição de Dados (*Shuffle*):** Como os registros contendo o mesmo `payment_type` (ex: tipo `1`, tipo `2`) estão espalhados por dezenas de partições diferentes do arquivo de 4M de linhas, o Spark é forçado a reorganizar os dados.
* **O Processo de Shuffle:**
  1. **Gravação em Disco Local (*Map Side*):** Cada partição processa seus dados locais, particiona os resultados pela chave de agrupamento e grava arquivos temporários no disco.
  2. **Transferência via Rede (*Network I/O*):** Os dados são enviados através da rede para que todas as linhas que possuem a *mesma chave* cheguem exatamente ao mesmo nó processador (*Reduce Side*).
  3. **Ordenação e Agregação (*Reduce Side*):** O nó de destino lê os dados recebidos da rede, realiza a ordenação/hashing na memória e só então calcula a soma e a contagem final.

---

## Conclusão e Resumo do Comparativo

* **`count()` / `filter()` / `select()`:** Trabalham com processamento paralelo local streaming, com **custo de rede praticamente zero**.
* **`groupBy()`:** Dispara um **Shuffle**, exigindo serialização de dados, gravações temporárias em disco (*spill*) e alto tráfego de rede para redistribuir milhões de linhas entre nós antes de consolidar os agregados. Por isso, operações com agrupamento e chaves de *join* são computacionalmente muito mais custosas.